<a href="https://colab.research.google.com/github/truegeo/python-async-programming/blob/main/Crash_Course_on_Python_Library_asyncio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Master Python `asyncio`: A Comprehensive Crash Course

Welcome to this interactive crash course on Python's `asyncio` library. I will be your guide through the world of asynchronous programming. By the end of this notebook, you will understand how to write highly concurrent I/O-bound code, manage background tasks, and avoid common pitfalls.

## 1. Introduction: What is Asyncio?

Before diving into code, let's understand the "why".

* **Synchronous Code:** Executes one line at a time. If line 2 takes 5 seconds to download a file, line 3 must wait 5 seconds.
* **Asynchronous Code:** While waiting for that file to download, the program says, "Hey, I'm waiting anyway, let me go do some other work."

`asyncio` is Python's built-in library to write **concurrent** code using the **async/await** syntax.

> ⚠️ **Crucial Concept:** Asyncio is generally for **I/O-bound** tasks (network requests, database queries, reading files). It is **not** for CPU-bound tasks (heavy math, image processing). For CPU-bound tasks, you want Multiprocessing.

### A Note on Jupyter Notebooks
Jupyter inherently runs an `asyncio` event loop in the background. Therefore, you can use `await` directly in standard cells. In a normal `.py` script, you would need to use `asyncio.run(main())` to start the program. We will use `await` here for interactive execution.

## 2. The Basics: Coroutines and `await`

The foundation of `asyncio` is the **coroutine**. You define a coroutine by putting `async` before `def`.

Calling an `async def` function **does not run it**. It returns a coroutine object. To actually run it, you must `await` it.

In [1]:
import asyncio
import time

async def say_hello():
    print("Hello...")
    # asyncio.sleep() is a non-blocking sleep. It simulates waiting for I/O.
    # Never use time.sleep() in async code! It blocks the whole event loop.
    await asyncio.sleep(1)
    print("...World!")
    return "Greeting finished"

# Notice that just calling it does nothing but create an object:
coro = say_hello()
print(f"Created: {coro}")

# Now we await it to execute it:
result = await coro
print(f"Result: {result}")

Created: <coroutine object say_hello at 0x7f097b41dcc0>
Hello...
...World!
Result: Greeting finished


## 3. Concurrent Execution: The Bread and Butter

If we just `await` one thing after another, we aren't saving any time. To achieve concurrency, we need to schedule multiple coroutines to run at the same time.

### `asyncio.gather()`
The most common way to run multiple coroutines concurrently and wait for all of them to finish.

In [2]:
async def fetch_data(id, delay):
    print(f"Task {id}: Starting fetch (takes {delay}s)...")
    await asyncio.sleep(delay)
    print(f"Task {id}: Finished!")
    return f"Data from {id}"

async def main_gather():
    start_time = time.time()

    # We schedule three tasks to run concurrently.
    # The total time will be ~3 seconds (the longest delay), NOT 1+2+3=6 seconds!
    results = await asyncio.gather(
        fetch_data(1, 1),
        fetch_data(2, 2),
        fetch_data(3, 3)
    )

    end_time = time.time()
    print(f"\nAll finished in {end_time - start_time:.2f} seconds!")
    print(f"Results: {results}")

await main_gather()

Task 1: Starting fetch (takes 1s)...
Task 2: Starting fetch (takes 2s)...
Task 3: Starting fetch (takes 3s)...
Task 1: Finished!
Task 2: Finished!
Task 3: Finished!

All finished in 3.00 seconds!
Results: ['Data from 1', 'Data from 2', 'Data from 3']


### `asyncio.create_task()` (Fire and Forget)

Sometimes you want to start a background task but you don't want to wait for it immediately. You can wrap a coroutine in a `Task`. It is scheduled to run on the event loop immediately.

In [3]:
async def background_logger():
    for i in range(3):
        print(f"[Background Logger] Logging heartbeat {i+1}...")
        await asyncio.sleep(0.5)

async def main_tasks():
    # Schedule the logger to run in the background
    task = asyncio.create_task(background_logger())

    print("[Main] Doing some important foreground work...")
    await asyncio.sleep(1.2) # Do work while logger runs
    print("[Main] Foreground work done!")

    # Wait for the background task to finish before exiting completely
    await task

await main_tasks()

[Main] Doing some important foreground work...
[Background Logger] Logging heartbeat 1...
[Background Logger] Logging heartbeat 2...
[Background Logger] Logging heartbeat 3...
[Main] Foreground work done!


## 4. Timeouts and Task Cancellation

In the real world, APIs hang and networks drop. You should never wait forever.

### `asyncio.wait_for()`
Wraps a coroutine and enforces a strict timeout.

In [4]:
async def slow_api_call():
    await asyncio.sleep(5) # Simulates a slow server
    return "Success!"

async def main_timeout():
    try:
        print("Calling API with a 2-second timeout...")
        # We only give it 2 seconds, but it needs 5. This will raise TimeoutError.
        result = await asyncio.wait_for(slow_api_call(), timeout=2.0)
        print(result)
    except asyncio.TimeoutError:
        print("Error: The API call took too long and was aborted!")

await main_timeout()

Calling API with a 2-second timeout...
Error: The API call took too long and was aborted!


## 5. Advanced Gathering: `as_completed`

What if you are downloading 10 images? You don't want to wait for all 10 to finish before showing the first one. `asyncio.as_completed()` yields coroutines as soon as they finish, in the order they finish.

In [5]:
import random

async def download_mock(filename):
    delay = random.uniform(0.5, 3.0)
    await asyncio.sleep(delay)
    return f"{filename} (took {delay:.2f}s)"

async def main_as_completed():
    tasks = [download_mock(f"file_{i}.txt") for i in range(1, 6)]

    print("Starting downloads...")
    # as_completed returns an iterator of coroutines
    for finished_task in asyncio.as_completed(tasks):
        result = await finished_task
        print(f"Downloaded instantly: {result}")

await main_as_completed()

Starting downloads...
Downloaded instantly: file_4.txt (took 0.68s)
Downloaded instantly: file_3.txt (took 1.30s)
Downloaded instantly: file_5.txt (took 1.40s)
Downloaded instantly: file_1.txt (took 1.45s)
Downloaded instantly: file_2.txt (took 2.40s)


## 6. Less Common (But Crucial) Synchronization Primitives

When multiple async tasks manipulate the same state, or when you need to limit resources, you use primitives very similar to standard threading.

### `asyncio.Semaphore()` (Rate Limiting)
Imagine you need to scrape 1,000 URLs. If you use `asyncio.gather()`, Python will fire off 1,000 requests instantly, crashing your computer or getting you banned by the server.

A **Semaphore** acts like a bouncer at a club. It only lets `N` tasks run at the same time.

In [6]:
async def rate_limited_worker(id, semaphore):
    # The 'async with' block waits if the semaphore is 'full' (currently 2)
    async with semaphore:
        print(f"Worker {id} acquired semaphore, doing work...")
        await asyncio.sleep(1) # Simulate work
        print(f"Worker {id} releasing semaphore.")

async def main_semaphore():
    # Only allow 2 concurrent operations at a time
    sem = asyncio.Semaphore(2)

    # Schedule 5 tasks. Notice how they process in chunks of 2!
    tasks = [asyncio.create_task(rate_limited_worker(i, sem)) for i in range(1, 6)]
    await asyncio.gather(*tasks)

await main_semaphore()

Worker 1 acquired semaphore, doing work...
Worker 2 acquired semaphore, doing work...
Worker 1 releasing semaphore.
Worker 2 releasing semaphore.
Worker 3 acquired semaphore, doing work...
Worker 4 acquired semaphore, doing work...
Worker 3 releasing semaphore.
Worker 4 releasing semaphore.
Worker 5 acquired semaphore, doing work...
Worker 5 releasing semaphore.


### `asyncio.Queue()` (Producer/Consumer Pattern)

Queues are the best way to pass data safely between different async tasks. A common pattern is having "producers" that find work, and "consumers" that process that work continuously.

In [7]:
async def producer(queue):
    for i in range(5):
        item = f"Item_{i}"
        await queue.put(item)
        print(f"[Producer] Added {item} to queue.")
        await asyncio.sleep(0.5)

async def consumer(queue):
    while True:
        item = await queue.get() # Waits until an item is available
        print(f"[Consumer] Processing {item}...")
        await asyncio.sleep(1) # Simulate processing time
        queue.task_done() # Tell the queue we are finished with this item

async def main_queue():
    queue = asyncio.Queue()

    # Fire up the consumer in the background (it runs forever)
    consumer_task = asyncio.create_task(consumer(queue))

    # Run the producer to fill the queue
    await producer(queue)

    # Wait until all items in the queue have been marked as task_done()
    await queue.join()

    # Clean up the infinite consumer task
    consumer_task.cancel()
    print("All work finished!")

await main_queue()

[Producer] Added Item_0 to queue.
[Consumer] Processing Item_0...
[Producer] Added Item_1 to queue.
[Consumer] Processing Item_1...
[Producer] Added Item_2 to queue.
[Producer] Added Item_3 to queue.
[Consumer] Processing Item_2...
[Producer] Added Item_4 to queue.
[Consumer] Processing Item_3...
[Consumer] Processing Item_4...
All work finished!


## 7. Bridging Sync and Async: The "Oh No" Button

**The Golden Rule of Asyncio:** NEVER block the event loop.

If you put `time.sleep(5)` or `requests.get('url')` inside an `async def` function, you freeze the *entire* async program for 5 seconds. No other tasks can run.

If you *must* use a synchronous, blocking library (like standard `requests` or heavy CPU-bound math) inside an async application, you use `asyncio.to_thread()` (introduced in Python 3.9). This pushes the blocking operation into a separate thread, freeing up the async event loop.

In [8]:
import time # The synchronous time module

def sync_blocking_function():
    print("  [Sync Thread] Starting long blocking task...")
    time.sleep(3) # THIS WOULD NORMALLY FREEZE ASYNCIO
    print("  [Sync Thread] Finished blocking task!")
    return "Sync Result"

async def async_counter():
    for i in range(4):
        print(f"[Async Loop] Ticking: {i}")
        await asyncio.sleep(1)

async def main_bridge():
    print("Running both blocking and non-blocking tasks...")

    # We wrap the synchronous function in to_thread so it doesn't block the loop
    await asyncio.gather(
        async_counter(),
        asyncio.to_thread(sync_blocking_function)
    )
    print("Complete!")

await main_bridge()

Running both blocking and non-blocking tasks...
[Async Loop] Ticking: 0
  [Sync Thread] Starting long blocking task...
[Async Loop] Ticking: 1
[Async Loop] Ticking: 2
  [Sync Thread] Finished blocking task!
[Async Loop] Ticking: 3
Complete!


## Summary

You have now covered the most vital parts of Python's `asyncio` ecosystem!

1.  **`async/await`**: The syntax to define and wait for asynchronous code.
2.  **`asyncio.gather`**: Wait for multiple things concurrently.
3.  **`asyncio.create_task`**: Push something to the background.
4.  **`asyncio.wait_for`**: Add strict timeouts to avoid hanging.
5.  **`asyncio.Semaphore`**: Protect yourself from doing too many things at once.
6.  **`asyncio.Queue`**: Safely hand off data between concurrent workers.
7.  **`asyncio.to_thread`**: A lifeline when you are forced to use synchronous blocking libraries in an async world.

Happy coding, and may your event loops never be blocked!

---
## Appendix A: Deep Dive into the Asyncio Library

As an expert async developer, you'll eventually need tools beyond the basics. Let's explore some of the modern and more intricate parts of the library.

### 1. Task Groups (Python 3.11+)
Introduced in Python 3.11, `TaskGroup` is the modern, safer alternative to `asyncio.gather()`. It acts as a context manager (`async with`). If any task inside the group fails with an exception, the `TaskGroup` automatically cancels all other running tasks inside it. This prevents "zombie tasks" from lingering in the background.

In [9]:
import sys

async def successful_task(id):
    await asyncio.sleep(1)
    return f"Result {id}"

async def task_group_demo():
    if sys.version_info < (3, 11):
        print("TaskGroups require Python 3.11 or higher. Skipping...")
        return

    print("Entering TaskGroup...")
    async with asyncio.TaskGroup() as tg:
        # We create tasks within the group.
        # We don't need to await them immediately; the 'async with' block waits for all of them!
        task1 = tg.create_task(successful_task(1))
        task2 = tg.create_task(successful_task(2))

    # Once the block exits, all tasks are guaranteed to be done.
    print(f"Tasks completed: {task1.result()} and {task2.result()}")

await task_group_demo()

Entering TaskGroup...
Tasks completed: Result 1 and Result 2


### 2. `asyncio.Event`: The Starting Pistol
An `Event` is a simple synchronization object. It holds a boolean flag (initially `False`). Multiple tasks can call `await event.wait()`, and they will all pause there. When another task calls `event.set()`, the flag becomes `True`, and **all** waiting tasks immediately wake up and continue.

In [10]:
async def racer(id, event):
    print(f"Racer {id} is at the starting line, waiting for the signal...")
    await event.wait()  # Block until the event is set
    print(f"⚡ Racer {id} is off!")

async def starter(event):
    print("Starter: 'On your marks...' ")
    await asyncio.sleep(1)
    print("Starter: 'Get set...' ")
    await asyncio.sleep(1)
    print("Starter: *BANG!* ")
    event.set() # Wake up all waiting racers!

async def event_demo():
    start_event = asyncio.Event()
    await asyncio.gather(
        racer("A", start_event),
        racer("B", start_event),
        starter(start_event)
    )

await event_demo()

Racer A is at the starting line, waiting for the signal...
Racer B is at the starting line, waiting for the signal...
Starter: 'On your marks...' 
Starter: 'Get set...' 
Starter: *BANG!* 
⚡ Racer A is off!
⚡ Racer B is off!


### 3. Asynchronous Subprocesses
If you need to run external command-line tools (like `ping`, `ls`, or a separate Python script), standard `subprocess.run()` will block the event loop. `asyncio` provides ways to run subprocesses asynchronously and read their output without freezing your program.

In [11]:
async def subprocess_demo():
    print("Running a shell command asynchronously...")
    # We use a shell command that echoes a string.
    # On Windows/Mac/Linux, 'echo' works universally.
    process = await asyncio.create_subprocess_shell(
        'echo Hello from the command line!',
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE
    )

    # wait for it to finish and read the standard output and error
    stdout, stderr = await process.communicate()

    if stdout:
        print(f"[Output]: {stdout.decode().strip()}")
    if stderr:
        print(f"[Error]: {stderr.decode().strip()}")

await subprocess_demo()

Running a shell command asynchronously...
[Output]: Hello from the command line!


---
## Appendix B: Real-World Use Cases

Theory is great, but let's look at how we combine these concepts into highly detailed, robust patterns you will use in actual production systems.

### Example 1: The Batch API Processor (Rate-Limited Fetching)
**Scenario:** You have an API that provides weather data, but it strictly limits you to 5 requests per second. You have 20 cities to look up. We must use a `Semaphore` to limit concurrency and `gather` to run the batches.

In [12]:
async def mock_weather_api(city, semaphore):
    async with semaphore:
        # The API request takes roughly 0.5 seconds
        await asyncio.sleep(0.5)
        # Simulate a network failure 10% of the time
        if random.random() < 0.1:
            return f"{city}: ERROR (Connection Lost)"
        return f"{city}: Sunny, 22°C"

async def example_1_rate_limited_api():
    cities = [f"City_{i}" for i in range(1, 21)]
    # Limit to exactly 5 simultaneous connections
    rate_limit = asyncio.Semaphore(5)

    print("Starting rate-limited batch processing...")
    start = time.time()

    # Create tasks for all cities
    tasks = [mock_weather_api(city, rate_limit) for city in cities]

    # Using as_completed so we can process successful results as they arrive,
    # rather than waiting for all 20 to finish.
    for future in asyncio.as_completed(tasks):
        result = await future
        print(f"  -> Received: {result}")

    total_time = time.time() - start
    print(f"Batch complete in {total_time:.2f} seconds. (Notice it took ~2 seconds total for 20 requests at 5/sec!)")

await example_1_rate_limited_api()

Starting rate-limited batch processing...
  -> Received: City_20: Sunny, 22°C
  -> Received: City_12: ERROR (Connection Lost)
  -> Received: City_1: Sunny, 22°C
  -> Received: City_13: Sunny, 22°C
  -> Received: City_5: Sunny, 22°C
  -> Received: City_14: ERROR (Connection Lost)
  -> Received: City_6: Sunny, 22°C
  -> Received: City_15: Sunny, 22°C
  -> Received: City_7: Sunny, 22°C
  -> Received: City_3: Sunny, 22°C
  -> Received: City_16: Sunny, 22°C
  -> Received: City_8: Sunny, 22°C
  -> Received: City_17: Sunny, 22°C
  -> Received: City_9: ERROR (Connection Lost)
  -> Received: City_18: Sunny, 22°C
  -> Received: City_4: Sunny, 22°C
  -> Received: City_10: Sunny, 22°C
  -> Received: City_19: Sunny, 22°C
  -> Received: City_11: Sunny, 22°C
  -> Received: City_2: ERROR (Connection Lost)
Batch complete in 2.00 seconds. (Notice it took ~2 seconds total for 20 requests at 5/sec!)


### Example 2: The Producer-Consumer Pipeline (Web Scraper)
**Scenario:** You are building a web scraper. One part of your program discovers URLs to scrape (Producer). Another part actually downloads and parses the text from those URLs (Consumer). A `Queue` is the perfect way to decouple them, allowing the producer to run ahead while multiple consumers chip away at the workload.

In [13]:
async def url_discoverer(queue):
    """Simulates finding links on a website (The Producer)"""
    urls_to_scrape = ["page1.html", "page2.html", "page3.html", "page4.html", "page5.html"]
    for url in urls_to_scrape:
        print(f"[Discoverer] Found link: {url}")
        await queue.put(url)
        await asyncio.sleep(0.2) # Finding links takes a little bit of time
    print("[Discoverer] Finished finding all links.")

async def html_parser(id, queue):
    """Simulates downloading and parsing the HTML (The Consumer)"""
    while True:
        # Wait for a URL to appear in the queue
        url = await queue.get()

        print(f"  [Parser {id}] Downloading {url}...")
        await asyncio.sleep(0.8) # Downloading/parsing is slow!
        print(f"  [Parser {id}] Finished parsing {url}!")

        # Crucial: Let the queue know we are done with this specific item
        queue.task_done()

async def example_2_pipeline():
    pipeline_queue = asyncio.Queue()

    # Spin up 3 parser "workers" running in the background
    consumers = [
        asyncio.create_task(html_parser(1, pipeline_queue)),
        asyncio.create_task(html_parser(2, pipeline_queue)),
        asyncio.create_task(html_parser(3, pipeline_queue))
    ]

    # Run the discoverer to populate the queue
    await url_discoverer(pipeline_queue)

    # Wait for the queue to be fully processed
    print("Waiting for parsers to finish the remaining queue items...")
    await pipeline_queue.join()

    # Shut down the background workers (otherwise they hang around forever waiting for queue.get())
    for c in consumers:
        c.cancel()
    print("Pipeline complete! All workers safely shut down.")

await example_2_pipeline()

[Discoverer] Found link: page1.html
  [Parser 1] Downloading page1.html...
[Discoverer] Found link: page2.html
  [Parser 2] Downloading page2.html...
[Discoverer] Found link: page3.html
  [Parser 3] Downloading page3.html...
[Discoverer] Found link: page4.html
  [Parser 1] Finished parsing page1.html!
  [Parser 1] Downloading page4.html...
[Discoverer] Found link: page5.html
  [Parser 2] Finished parsing page2.html!
  [Parser 2] Downloading page5.html...
[Discoverer] Finished finding all links.
Waiting for parsers to finish the remaining queue items...
  [Parser 3] Finished parsing page3.html!
  [Parser 1] Finished parsing page4.html!
  [Parser 2] Finished parsing page5.html!
Pipeline complete! All workers safely shut down.


### Example 3: Graceful Shutdowns with Background Monitors
**Scenario:** You have a main program running (like a web server or game loop). Alongside it, you want a background task that constantly monitors system health. When the main program shuts down, the background monitor needs a polite signal to clean up and exit, rather than just being abruptly killed. We use `asyncio.Event` for this.

In [14]:
async def system_monitor(shutdown_event):
    """Runs in the background until the shutdown_event is triggered."""
    print("[Monitor] Starting background health checks...")

    # We loop as long as the event is NOT set
    while not shutdown_event.is_set():
        print("  [Monitor] System is healthy. ✅")

        # We use wait_for with a timeout as a way to sleep.
        # If the shutdown_event is set during this 1 second, it immediately breaks the wait!
        try:
            await asyncio.wait_for(shutdown_event.wait(), timeout=1.0)
        except asyncio.TimeoutError:
            # Timeout simply means the event wasn't set. Continue the loop.
            pass

    print("[Monitor] Shutdown signal received! Cleaning up resources... ⛔")
    await asyncio.sleep(0.5) # Simulate cleanup
    print("[Monitor] Shut down safely.")

async def example_3_graceful_shutdown():
    shutdown_signal = asyncio.Event()

    # Start the monitor in the background
    monitor_task = asyncio.create_task(system_monitor(shutdown_signal))

    print("[Main] Server is running...")
    await asyncio.sleep(3.5) # The main application doing its work for a few seconds
    print("[Main] Server is stopping! Signalling background tasks...")

    # Trigger the event to let the background task know it's time to stop
    shutdown_signal.set()

    # Await the task to ensure it has time to finish its cleanup
    await monitor_task
    print("[Main] Complete shutdown successful.")

await example_3_graceful_shutdown()

[Main] Server is running...
[Monitor] Starting background health checks...
  [Monitor] System is healthy. ✅
  [Monitor] System is healthy. ✅
  [Monitor] System is healthy. ✅
  [Monitor] System is healthy. ✅
[Main] Server is stopping! Signalling background tasks...
[Monitor] Shutdown signal received! Cleaning up resources... ⛔
[Monitor] Shut down safely.
[Main] Complete shutdown successful.
